# GXAI-Elderly: Unified Training Notebook

This notebook trains all three layers of the GXAI-Elderly framework on the **full COVA-X dataset** (10,985 conversations).

**Layers:**
1. **Layer 1** – BERT/RoBERTa scam type classifier
2. **Layer 2** – Sentence-Transformers + BiLSTM cognitive profiler
3. **Layer 3** – LoRA-fine-tuned LLM gist generator

**Hardware:** Designed for Google Colab free T4 GPU (16 GB VRAM).

**Before you start:** Upload your `Research in Aging` folder (with the dataset) to the root of your Google Drive.

In [ ]:
# ==============================================================================
# CELL 1: Mount Google Drive
# ==============================================================================
from google.colab import drive
drive.mount('/content/drive')

# Change to your project root (update the path if you placed it elsewhere)
PROJECT_ROOT = '/content/drive/MyDrive/Research in Aging'
%cd $PROJECT_ROOT
!pwd

In [ ]:
# ==============================================================================
# CELL 2: Install Dependencies
# ==============================================================================
!pip install -q -r paper1_gxai_elderly/experiments/requirements.txt

# (Optional) Login to Hugging Face if you plan to use gated models such as Llama-3.1-8B or Mistral-7B
# Uncomment the next two lines and paste your token when prompted.
# !pip install -q huggingface_hub
# from huggingface_hub import notebook_login; notebook_login()

In [ ]:
# ==============================================================================
# CELL 3: Verify Dataset
# ==============================================================================
import os, glob, json

base_dir = 'dataset/COVA-X Dataset/COVA-X_Dataset'
categories = [d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))]
total = 0
for cat in categories:
    n = len(glob.glob(os.path.join(base_dir, cat, '*.json')))
    print(f"{cat:30s}: {n} conversations")
    total += n
print(f"\nTotal conversations: {total}")

---
## Layer 1: Scam Type Classification (BERT)
*Default model:* `distilbert-base-uncased` (~66 M params).
*Expected time on Colab T4:* ~20–30 min for 5 epochs.
*Output:* `./layer1_bert_results/best_model`

In [ ]:
!python paper1_gxai_elderly/experiments/train_layer1_bert.py

---
## Layer 2: Cognitive Profiler (Sentence-Transformers + BiLSTM)
*Predicts:* `trust_level`, `tech_savviness`, `scam_awareness`.
*Expected time on Colab T4:* ~15–25 min (all three targets).
*Output:* `./layer2_profiler_results/`*`_best.pt`

In [ ]:
!python paper1_gxai_elderly/experiments/train_layer2_profiler.py

---
## Layer 3: Gist Translation Engine (LoRA Fine-Tuned LLM)
*Default model:* `TinyLlama/TinyLlama-1.1B-Chat-v1.0` (fits free Colab T4).
*For stronger results* switch to `meta-llama/Meta-Llama-3.1-8B-Instruct` or `mistralai/Mistral-7B-Instruct-v0.3` (requires HuggingFace token + 16 GB VRAM).
*Expected time on Colab T4:* ~30–45 min for 3 epochs.
*Output:* `./layer3_llm_lora_results/lora_adapter`

In [ ]:
# To use a gated model, uncomment the MODEL_NAME line inside the script or set an env var.
# Example for Llama-3.1-8B (requires HuggingFace token):
# !export COVA_X_LLM_MODEL="meta-llama/Meta-Llama-3.1-8B-Instruct" && python paper1_gxai_elderly/experiments/train_layer3_llm_lora.py

!python paper1_gxai_elderly/experiments/train_layer3_llm_lora.py

---
## End-to-End Inference
Run the full pipeline (Layer 1 → Layer 2 → Layer 3) on a single conversation or a batch.

In [ ]:
# Single conversation demo
sample = 'dataset/COVA-X Dataset/COVA-X_Dataset/bank/bank_001.json'  # update path if needed
!python paper1_gxai_elderly/experiments/evaluate_end_to_end.py --conversation "$sample"

In [ ]:
# Batch evaluation (first 50 conversations)
!python paper1_gxai_elderly/experiments/evaluate_end_to_end.py --test_dir "dataset/COVA-X Dataset/COVA-X_Dataset" --limit 50

---
## Save / Backup Results to Drive
All model checkpoints are already saved in the project directory on Drive.
If you want to zip them for download:

In [ ]:
!zip -r layer1_bert_results.zip layer1_bert_results/best_model
!zip -r layer2_profiler_results.zip layer2_profiler_results
!zip -r layer3_lora_adapter.zip layer3_llm_lora_results/lora_adapter
print("Zipped models saved in project root on Google Drive.")

---
## Next Steps
1. **Check metrics** printed after each layer.
2. **Download** the zipped models if you need local copies.
3. **Switch to Llama-3.1-8B** for Layer 3 by editing `MODEL_NAME` in `train_layer3_llm_lora.py` and re-running.
4. **Use the evaluation outputs** (`end_to_end_results.json`) for paper tables and figures.